# Evaluation of Splines
The obects that the class ``PeriodicSpline1D`` operates upon are mathematical functions and consist in descriptions of mappings $f:{\mathbb{R}}\rightarrow{\mathbb{R}},x\mapsto f(x).$ Oftentimes, the class allows one to obtain a new *mapping* from an existing one, for instance the gradient of a spline (not just a number but the whole function $\dot{f}$) out of its original form $f.$ This is all good, but one is sometimes also interested in the more mundane goal of experimenting with a fixed mapping, typically to ask to what number $f(x)\in{\mathbb{R}}$ is the argument $x\in{\mathbb{R}}$ mapped to by $f:{\mathbb{R}}\rightarrow{\mathbb{R}}.$ The ``splinekit`` library  offers two possibilities to address this goal.
*   ``PeriodicSpline1D.at`` returns the value $f(x)$ of the spline $f$ evaluated at $x.$
*   ``PeriodicSpline1D.get_samples`` returns an array of values at arguments separated by a uniform step.

The two methods rely on the recipe followed by all one-dimensional uniform polynomial splines of nonnegative integer degree $n\in{\mathbb{N}},$ according to which

$$f(x)=\sum_{k\in{\mathbb{Z}}}\,c[{k\bmod K}]\,\beta^{n}(x-\delta x -k),$$
where $K\in{\mathbb{N}}+1$ is a positive integer period, $c$ is an arbitrary array of $K$ spline coefficients, $\beta^{n}$ is a B-spline whose degree $n$ is typeset in superscript (not a power), and $\delta x\in{\mathbb{R}}$ is an arbitrary delay. Put simply, a spline is a weighted sum of shifted B-splines.

Since the support of a B-spline is finite, the sum is finite at any given argument $x.$ Moreover, since B-splines of positive degrees are themselves made of unit-length pieces of polynomials, one can deploy the formalism of linear-algebra to express the spline recipe as

$$\begin{eqnarray*}
\forall n\in{\mathbb{N}}+1,\forall x\in{\mathbb{R}}:f(x)&=&{\mathbf{c}}^{{\mathsf{T}}}\,{\mathbf{W}}^{n}\,{\mathbf{v}}^{n}(x-\delta x)\\
&=&\left(\begin{array}{c}c[{\left(-m\right)\bmod K}]\\c[{\left(1-m\right)\bmod K}]\\c[{\left(2-m]\right)\bmod K}\\\vdots\\c[{\left(n-m\right)\bmod K}]\end{array}\right)^{{\mathsf{T}}}\,\left(\begin{array}{ccccc}w_{0,0}^{n}&w_{0,1}^{n}&w_{0,2}^{n}&\cdots&w_{0,n}^{n}\\w_{1,0}^{n}&w_{1,1}^{n}&w_{1,2}^{n}&\cdots&w_{1,n}^{n}\\w_{2,0}^{n}&w_{2,1}^{n}&w_{2,2}^{n}&\cdots&w_{2,n}^{n}\\\vdots&\vdots&\vdots&\ddots&\vdots\\w_{n,0}^{n}&w_{n,1}^{n}&w_{n,2}^{n}&\cdots&w_{n,n}^{n}\end{array}\right)\,\left(\begin{array}{c}1\\v\\v^{2}\\\vdots\\v^{n}\end{array}\right),
\end{eqnarray*}$$
where ${\mathbf{c}}\in{\mathbb{R}}^{n+1}$ is a vector whose $n+1$ components are extracted from the data-dependent array $c$ at some integer but $\left(x-\delta x\right)$-dependent $m\in{\mathbb{Z}},$ where ${\mathbf{W}}\in{\mathbb{R}}^{\left(n+1\right)\times\left(n+1\right)}$ is a matrix that depends on $n$ only, and where ${\mathbf{v}}^{n}$ is a Vandermonde vector that depends continuously on $\left(x-\delta x\right)$ and whose every component belongs to the interval $[0,1].$



## Single Argument

In [1]:
# Load the required libraries.
from IPython.display import display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

import splinekit as sk # This library

# Setup
period = 8 # Support of the data samples
max_degree = 5 # Maximal spline degree
max_delay = 5.0 # Maximal absolute delay

# Persistent spline
f = sk.PeriodicSpline1D()

# Initialize the generator of random numbers
rng = np.random.default_rng()

# Widgets
degree = widgets.IntSlider(value = 3, min = 0, max = max_degree, )
delay = widgets.FloatSlider(value = 0.0, min = -max_delay, max = max_delay)
x = widgets.FloatSlider(value = 0.5 * period, min = -1.0, max = period + 1.0)
fx = widgets.Label(value = "")

# Plot
def update_plot (
    degree,
    delay,
    x
):
    global f # This spline

    # Check whether new data are needed
    if degree != f.degree:
        c = rng.standard_normal(period) # Fresh data
        f = sk.PeriodicSpline1D.from_spline_coeff(c, degree = degree)
    f.delay = delay
    fx.value = "$f({0:.2f}) = {1:.4f}$".format(x, f.at(x))
    # Plot canvas
    (fig, ax) = plt.subplots()
    # Spline in plain style
    f.plot(
        (fig, ax),
        plotpoints = 301,
        curve_markerfmt = " ",
        knot_marker = " ",
        periodbound_markerfmt = " "
    )
    ax.plot([x], [f.at(x)], "o")
    plt.show()

display(
    widgets.VBox([
        widgets.HBox([
            widgets.Label(value = "Degree", layout = widgets.Layout(width = "36px")),
            degree
        ]),
        widgets.HBox([
            widgets.Label(value = "Delay", layout = widgets.Layout(width = "36px")),
            delay
        ]),
        widgets.HBox([
            widgets.Label(value = "x", layout = widgets.Layout(width = "36px")),
            x
        ]),
        fx
    ]),
    widgets.interactive_output(update_plot, {'degree': degree, 'delay': delay, 'x': x})
)


Output()

## Multiple Arguments